
## Folder 01 — Environment Setup and Imports

In [0]:
## 1. Import required libraries

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import *
from delta.tables import DeltaTable

from datetime import datetime, timezone
import uuid
import logging

print("Required libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "data_volume"

PROJECT_NAME = "ecommerce_lakehouse"


# Main project storage path

BASE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{PROJECT_NAME}"

# Project directories

LANDING_PATH = f"{BASE_PATH}/landing"
CHECKPOINT_PATH = f"{BASE_PATH}/checkpoints"
SCHEMA_PATH = f"{BASE_PATH}/schemas"
LOG_PATH = f"{BASE_PATH}/logs"
QUARANTINE_PATH = f"{BASE_PATH}/quarantine"


# Medallion table schemas

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

print("Project configuration loaded.")


## 3. Dataset configuration


DATASETS = [
    "customers",
    "products",
    "orders",
    "order_items",
    "payments",
    "shipments",
    "inventory_snapshots"
]


# Primary keys for each dataset

PRIMARY_KEYS = {
    "customers": ["customer_id"],
    "products": ["product_id"],
    "orders": ["order_id"],
    "order_items": ["order_item_id"],
    "payments": ["payment_id"],
    "shipments": ["shipment_id"],
    "inventory_snapshots": ["inventory_snapshot_id"]
}


# Expected source CSV columns

EXPECTED_COLUMNS = {
    "customers": [
        "customer_id", "first_name", "last_name", "email",
        "phone", "city", "state", "created_at"
    ],

    "products": [
        "product_id", "product_name", "category", "unit_price",
        "unit_cost", "stock_status", "created_at"
    ],

    "orders": [
        "order_id", "customer_id", "order_timestamp",
        "order_status", "currency", "order_total", "sales_channel"
    ],

    "order_items": [
        "order_item_id", "order_id", "product_id", "quantity",
        "unit_price", "discount_rate", "line_total"
    ],

    "payments": [
        "payment_id", "order_id", "payment_method",
        "payment_status", "amount", "payment_timestamp",
        "transaction_id"
    ],

    "shipments": [
        "shipment_id", "order_id", "carrier", "shipment_status",
        "shipped_date", "delivered_date", "tracking_number"
    ],

    "inventory_snapshots": [
        "inventory_snapshot_id", "product_id",
        "quantity_on_hand", "snapshot_date", "warehouse_id"
    ]
}

print(f"Total datasets configured: {len(DATASETS)}")


## 4. Validate Unity Catalog resources

# Validate catalog


catalogs = [
    row.catalog
    for row in spark.sql("SHOW CATALOGS").collect()
]

if CATALOG not in catalogs:
    raise Exception(f"Catalog '{CATALOG}' does not exist.")

print(f"Catalog validated: {CATALOG}")


# Validate schema

schemas = [
    row.databaseName
    for row in spark.sql(
        f"SHOW SCHEMAS IN `{CATALOG}`"
    ).collect()
]

if SCHEMA not in schemas:
    raise Exception(
        f"Schema '{SCHEMA}' does not exist in catalog '{CATALOG}'."
    )

print(f"Schema validated: {CATALOG}.{SCHEMA}")


# Validate volume

spark.sql(
    f"DESCRIBE VOLUME `{CATALOG}`.`{SCHEMA}`.`{VOLUME}`"
).collect()

print(f"Volume validated: {CATALOG}.{SCHEMA}.{VOLUME}")


## 5. Create project directories


PROJECT_DIRECTORIES = {
    "landing": LANDING_PATH,
    "checkpoints": CHECKPOINT_PATH,
    "schemas": SCHEMA_PATH,
    "logs": LOG_PATH,
    "quarantine": QUARANTINE_PATH
}


# Create main directories

for directory_name, directory_path in PROJECT_DIRECTORIES.items():
    dbutils.fs.mkdirs(directory_path)
    print(f"Created/verified: {directory_path}")


# Create individual dataset landing directories

for dataset in DATASETS:
    dataset_path = f"{LANDING_PATH}/{dataset}"
    dbutils.fs.mkdirs(dataset_path)
    print(f"Created/verified: {dataset_path}")

print("Project directory structure initialized.")


## 6. Define Delta table names


BRONZE_TABLES = {
    dataset: f"{CATALOG}.{BRONZE_SCHEMA}.bronze_{dataset}"
    for dataset in DATASETS
}

SILVER_TABLES = {
    dataset: f"{CATALOG}.{SILVER_SCHEMA}.silver_{dataset}"
    for dataset in DATASETS
}

GOLD_TABLES = {
    "daily_sales": f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales",
    "product_performance": f"{CATALOG}.{GOLD_SCHEMA}.gold_product_performance",
    "customer_lifetime_value": f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_lifetime_value",
    "payment_summary": f"{CATALOG}.{GOLD_SCHEMA}.gold_payment_summary",
    "shipment_sla": f"{CATALOG}.{GOLD_SCHEMA}.gold_shipment_sla",
    "inventory_latest": f"{CATALOG}.{GOLD_SCHEMA}.gold_inventory_latest"
}

print("Bronze, Silver, and Gold table names configured.")


## 7. Create pipeline execution metadata


EXECUTION_ID = str(uuid.uuid4())

PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Execution ID: {EXECUTION_ID}")
print(f"Pipeline start time: {PIPELINE_START_TIME}")


## 8. Environment summary


print("=" * 60)
print("E-COMMERCE LAKEHOUSE ENVIRONMENT")
print("=" * 60)

print(f"Catalog          : {CATALOG}")
print(f"Schema           : {SCHEMA}")
print(f"Volume           : {VOLUME}")
print(f"Base Path        : {BASE_PATH}")

print("-" * 60)

print(f"Landing Path     : {LANDING_PATH}")
print(f"Checkpoint Path  : {CHECKPOINT_PATH}")
print(f"Schema Path      : {SCHEMA_PATH}")
print(f"Log Path         : {LOG_PATH}")
print(f"Quarantine Path  : {QUARANTINE_PATH}")

print("-" * 60)

print(f"Total Datasets   : {len(DATASETS)}")
print(f"Execution ID     : {EXECUTION_ID}")

print("=" * 60)
print("ENVIRONMENT SETUP COMPLETED SUCCESSFULLY")
print("=" * 60)



## Lakehouse directory and schema initialization:


**Run this in a Databricks Python notebook:**

In [0]:

# ============================================================
# E-COMMERCE LAKEHOUSE
# ENVIRONMENT SETUP AND DIRECTORY INITIALIZATION
# ============================================================


# COMMAND 1: CONFIGURATION

CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"
    "ecommerce_lakehouse"
)

print("E-Commerce Lakehouse Environment Setup")
print("-" * 60)

print(f"Catalog       : {CATALOG}")
print(f"Volume Schema : {SCHEMA}")
print(f"Volume        : {VOLUME}")
print(f"Base Path     : {BASE_PATH}")


# COMMAND 2: IMPORT REQUIRED LIBRARIES

from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

print("\nLibraries imported successfully.")


# COMMAND 3: DEFINE PROJECT DIRECTORIES

directories = [

    # --------------------------------------------------------
    # LANDING LAYER
    # --------------------------------------------------------

    "landing/customers",
    "landing/products",
    "landing/orders",
    "landing/order_items",
    "landing/payments",
    "landing/shipments",
    "landing/inventory_snapshots",


    # --------------------------------------------------------
    # BRONZE DELTA STORAGE
    # --------------------------------------------------------

    "bronze/customers",
    "bronze/products",
    "bronze/orders",
    "bronze/order_items",
    "bronze/payments",
    "bronze/shipments",
    "bronze/inventory_snapshots",


    # --------------------------------------------------------
    # SILVER DELTA STORAGE
    # --------------------------------------------------------

    "silver/customers",
    "silver/products",
    "silver/orders",
    "silver/order_items",
    "silver/payments",
    "silver/shipments",
    "silver/inventory_snapshots",


    # --------------------------------------------------------
    # GOLD DELTA STORAGE
    # --------------------------------------------------------

    "gold/daily_sales",
    "gold/product_performance",
    "gold/customer_lifetime_value",
    "gold/payment_metrics",
    "gold/shipping_metrics",
    "gold/inventory_status",
    "gold/executive_kpis",


    # --------------------------------------------------------
    # CHECKPOINTS
    # --------------------------------------------------------

    "checkpoints/bronze/customers",
    "checkpoints/bronze/products",
    "checkpoints/bronze/orders",
    "checkpoints/bronze/order_items",
    "checkpoints/bronze/payments",
    "checkpoints/bronze/shipments",
    "checkpoints/bronze/inventory_snapshots",

    "checkpoints/silver",
    "checkpoints/checkpoint_backups",


    # --------------------------------------------------------
    # AUTO LOADER SCHEMAS
    # --------------------------------------------------------

    "schemas/bronze/customers",
    "schemas/bronze/products",
    "schemas/bronze/orders",
    "schemas/bronze/order_items",
    "schemas/bronze/payments",
    "schemas/bronze/shipments",
    "schemas/bronze/inventory_snapshots",


    # --------------------------------------------------------
    # LOGS
    # --------------------------------------------------------

    "logs/bronze",
    "logs/silver",
    "logs/gold",
    "logs/pipeline",


    # --------------------------------------------------------
    # QUARANTINE
    # --------------------------------------------------------

    "quarantine/bronze",
    "quarantine/silver",
    "quarantine/data_quality",
    "quarantine/referential_integrity"
]


# COMMAND 4: CREATE PROJECT DIRECTORIES

print("\nCreating project directories...")
print("-" * 60)

for directory in directories:

    path = f"{BASE_PATH}/{directory}"

    dbutils.fs.mkdirs(path)

    print(f"Created/verified: {path}")

print("\nAll project directories created successfully.")


# COMMAND 5: CREATE UNITY CATALOG SCHEMAS

schemas = [
    "ecommerce_bronze",
    "ecommerce_silver",
    "ecommerce_gold"
]

print("\nCreating Unity Catalog schemas...")
print("-" * 60)

for schema_name in schemas:

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {CATALOG}.{schema_name}
        """
    )

    print(f"Schema ready: {CATALOG}.{schema_name}")


# COMMAND 6: VALIDATE PROJECT DIRECTORIES

print("\nValidating project directories...")
print("-" * 60)

validation_results = []

for directory in directories:

    path = f"{BASE_PATH}/{directory}"

    try:

        dbutils.fs.ls(path)

        status = "SUCCESS"

    except Exception as e:

        status = "FAILED"

    validation_results.append(
        (directory, path, status)
    )

validation_df = spark.createDataFrame(
    validation_results,
    ["Directory", "Physical_Path", "Status"]
)

display(validation_df)


# COMMAND 7: VALIDATE UNITY CATALOG SCHEMAS

print("\nValidating Unity Catalog schemas...")
print("-" * 60)

display(
    spark.sql(f"SHOW SCHEMAS IN {CATALOG}")
)


# COMMAND 8: DISPLAY FINAL STORAGE LOCATIONS

print("\nFINAL LAKEHOUSE STORAGE LOCATIONS")
print("=" * 60)

storage_layers = [
    "landing",
    "bronze",
    "silver",
    "gold",
    "checkpoints",
    "schemas",
    "logs",
    "quarantine"
]

for layer in storage_layers:

    print(
        f"{layer.upper():15} : "
        f"{BASE_PATH}/{layer}"
    )


# COMMAND 9: FINAL INITIALIZATION STATUS

total_directories = len(directories)

successful_directories = sum(
    1 for item in validation_results
    if item[2] == "SUCCESS"
)

print("\n" + "=" * 60)
print("E-COMMERCE LAKEHOUSE INITIALIZATION SUMMARY")
print("=" * 60)

print(f"Total directories       : {total_directories}")
print(f"Successfully validated : {successful_directories}")
print(f"Failed directories     : {total_directories - successful_directories}")
print(f"Unity Catalog schemas  : {len(schemas)}")

if successful_directories == total_directories:

    print("\nSUCCESS: All directories validated.")

else:

    print("\nWARNING: Some directories failed validation.")

print("\nEnvironment setup completed.")
